# 06 · Linear GAM (MOS)

**Input:** `data/processed/*.parquet` (from `scripts/build_dataset.py`) and the raw ECMWF runs.
**Output:** `reports/predictions/linear_gam.csv` (and `lr_mos.csv`), the hourly forecast for 12am Sep 17 – 11pm Sep 30, 2026.

Same approach as the linear-regression baseline (PR #3), on this repo's pipeline:

- **Target = anomaly**: observed − climatology (`clim_temp_c`). Forecast = climatology + predicted anomaly.
- **One model per lead day** (1–14), so each day learns how far to trust ECMWF.
- **Recency weights**, half-life 180 days, because ECMWF's bias at RDU drifts.
- **Same 12 inputs** as the LR: ECMWF anomaly and time-lagged anomaly, ECMWF's error over the last 6 observed hours, its 14/30/60-day bias, the station's current and 24-hour anomaly, and the hour.

**What changes:** each input enters through a smooth curve instead of a straight line (`pygam.LinearGAM`: penalised B-splines, identity link, smoothness chosen by GCV per lead day). The hour is one cyclic spline instead of four sin/cos terms; the three slow-moving bias terms stay linear.

`lr_mos` is the LR rebuilt on the same rows and inputs, so the comparison isolates *linear vs smooth*. The code lives in `src/weather_modeling/gam.py`; `scripts/evaluate_linear_gam.py` runs Sections 2–4 in one go.

Setup: `pip install -r requirements.txt` (adds `pygam`), then `pip install -e .`.

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from weather_modeling import config
from weather_modeling.evaluation import BASELINES, cross_validate, make_submission, plot_by_lead_day, score_folds
from weather_modeling.gam import (
    GAM_FEATURES, LR_FEATURES, TASK_WINDOWS_2026, build_mos_table, ecmwf_weight, fit_gam, fit_lr,
    forecast_rows, gam_fit_predict, lr_fit_predict, predict, trainable,
)
from weather_modeling.splits import FOLDS

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
MODELS = {"lr_mos": lr_fit_predict, "linear_gam": gam_fit_predict}
NAMES = [*BASELINES, *MODELS]

## 1. Inputs

`build_mos_table()` adds the LR's inputs to `forecast_pairs.parquet`. Every input uses only the run itself, earlier 12z runs, and observations reported before the run's cutoff (`tests/test_gam.py` rebuilds runs from truncated data and checks nothing moves).

Differences from the LR notebook, because this pipeline differs: the base is the 2015–2023 climatology (`clim_temp_c`) rather than an expanding "earlier years" average; the time-lagged ensemble uses the 12z runs of days *d*, *d*−1, *d*−2 (only 12z runs are archived here, so no 00z member).

In [ ]:
table = build_mos_table()
data = trainable(table)
final_rows = forecast_rows(table)
print(f"{len(data):,} trainable rows from {data['run_time_utc'].nunique()} runs; {len(final_rows)} rows to forecast")
display(data[LR_FEATURES].describe().T[["mean", "std", "min", "max"]].round(2))

## 2. Validation: the two shared folds

Each fold trains only on targets observed before its first validation target (purged split), then every model is scored on the same hours. `all` weights the 14 lead days equally (the framework's default); the second table also shows hours pooled, the convention of the LR's numbers in its PR.

In [ ]:
scored = cross_validate(MODELS, data, FOLDS)
mae = score_folds(scored, NAMES)
display(mae.round(2))

pooled = score_folds(scored, NAMES, overall="pooled").xs("all", level="lead_day")[NAMES]
equal = mae.xs("all", level="lead_day")[NAMES]
display(pd.concat({"lead days equal": equal, "hours pooled": pooled}, names=["weighting"]).round(2))
display(pd.DataFrame({b: 100 * (1 - equal["linear_gam"] / equal[b]) for b in ["persistence", "climatology", "raw_ecmwf", "lr_mos"]})
        .round(0).add_prefix("% lower MAE than "))

In [ ]:
for fold, part in mae.groupby(level="fold", sort=False):
    plot_by_lead_day(part.droplevel("fold")[["climatology", "raw_ecmwf", "persistence", *MODELS, "n"]],
                     f"MAE by lead day, {fold}", path=config.FIGURES_DIR / f"linear_gam_mae_{fold}.png")
    plt.show()

## 3. Extra check: six 2026 windows

Six single forecasts set up exactly like the real task, each from a model retrained on data before that window.

In [ ]:
windows = cross_validate(MODELS, data, TASK_WINDOWS_2026)
window_mae = score_folds(windows, NAMES, overall="pooled").xs("all", level="lead_day")[NAMES]
window_mae.loc["Average"] = window_mae.mean()
display(window_mae.round(2))

## 4. Final models and the forecast for Sep 17–30

Trained on every row with a target (all observed before 12am Sep 17). The forecast uses the Sep 16 12z run and observations up to 11:51pm Sep 16. The files are fingerprinted (SHA-256) **before** any test scoring.

In [ ]:
gam_models, lr_models = fit_gam(data), fit_lr(data)
forecast = final_rows[["target_time_local", "clim_temp_c", "ecmwf_t2m"]].copy()
forecast["linear_gam"] = predict(gam_models, final_rows, GAM_FEATURES)
forecast["lr_mos"] = predict(lr_models, final_rows, LR_FEATURES)
assert forecast[["linear_gam", "lr_mos"]].notna().all().all() and len(forecast) == 336

frozen = []
for name in MODELS:
    path = make_submission(forecast[name], name)
    frozen.append({"name": name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                   "frozen_at_utc": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M:%S")})
frozen = pd.DataFrame(frozen)
frozen.to_csv(config.PREDICTIONS_DIR / "frozen_forecasts.csv", index=False)
display(frozen)

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(forecast["target_time_local"], forecast["clim_temp_c"], "--", lw=1, color="#2a78d6", label="climatology")
ax.plot(forecast["target_time_local"], forecast["ecmwf_t2m"], lw=1, color="#1baf7a", label="raw ECMWF (Sep 16 12z)")
ax.plot(forecast["target_time_local"], forecast["lr_mos"], lw=1.2, color="#4a3aa7", label="lr_mos")
ax.plot(forecast["target_time_local"], forecast["linear_gam"], lw=1.8, color="#008300", label="linear_gam")
ax.set(title="Forecast for RDU, Sep 17-30, 2026 (local time)", ylabel="°C")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "linear_gam_final_forecast.png", dpi=150)
plt.show()

### 4.1 What the GAM learned

**Partial dependence**: the contribution of each input to the predicted anomaly, with a 95% band, for three lead days. Straight lines here mean the LR's form was already right for that input; bends are what the GAM adds.

In [ ]:
SHOW_DAYS = (1, 7, 14)
terms = [i for i, term in enumerate(gam_models[1].terms) if not term.isintercept]
fig, axes = plt.subplots(len(SHOW_DAYS), len(terms), figsize=(2.3 * len(terms), 2.2 * len(SHOW_DAYS)), sharey="row")
for row, day in zip(axes, SHOW_DAYS):
    gam = gam_models[day]
    for ax, i in zip(row, terms):
        grid = gam.generate_X_grid(term=i)
        effect, band = gam.partial_dependence(term=i, X=grid, width=0.95)
        x = grid[:, gam.terms[i].feature]
        ax.plot(x, effect, color="#008300")
        ax.fill_between(x, band[:, 0], band[:, 1], color="#008300", alpha=0.15, lw=0)
        ax.axhline(0, color="grey", lw=0.5)
        if day == SHOW_DAYS[0]:
            ax.set_title(GAM_FEATURES[gam.terms[i].feature], fontsize=8)
    row[0].set_ylabel(f"day {day}\n°C")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "linear_gam_partial_dependence.png", dpi=150)
plt.show()

summary = pd.DataFrame({day: {"lambda": float(np.ravel(m.lam)[0]),
                              "effective dof": m.statistics_["edof"], "GCV": m.statistics_["GCV"]}
                        for day, m in gam_models.items()}).T
display(summary.round(3))

**How much the model follows ECMWF.** For the GAM: the average change of the prediction when both ECMWF anomaly inputs rise by 1 °C, over the last year of rows. For the LR: its two ECMWF coefficients added up (as in the LR notebook's Section 5.1).

In [ ]:
last_year = data[data["run_time_utc"] >= data["run_time_utc"].max() - pd.Timedelta(days=365)]
trust = pd.DataFrame({
    "linear_gam": ecmwf_weight(gam_models, last_year),
    "lr_mos": pd.Series({d: m.coef_[0] + m.coef_[1] for d, m in lr_models.items()}),
}).T
trust.columns.name = "lead day"
display(trust.round(2))

## 5. Test period: Sep 17–30 (run once, after Section 4)

The forecasts above are frozen. This cell scores them, with the baselines, against the observed temperatures through `scripts/score_submission.py`, which logs each file's SHA-256 to `reports/final_scoring_log.csv` and warns if a forecast changed after it was first scored. Do not tune anything against these numbers.

`python scripts/download_data.py --holdout` refreshes the observations first if NOAA has published more of the 336 hours.

In [ ]:
import subprocess, sys

frozen_now = {p.stem: hashlib.sha256(p.read_bytes()).hexdigest() for p in
              [config.PREDICTIONS_DIR / f"{name}.csv" for name in MODELS]}
assert frozen_now == dict(zip(frozen["name"], frozen["sha256"])), "forecast changed since it was frozen"

result = subprocess.run([sys.executable, str(config.PROJECT_ROOT / "scripts" / "score_submission.py"),
                         *[str(config.PREDICTIONS_DIR / f"{name}.csv") for name in MODELS]],
                        capture_output=True, text=True, cwd=config.PROJECT_ROOT)
print(result.stdout, result.stderr)

In [ ]:
from weather_modeling.evaluation import holdout_frame
from weather_modeling.metrics import score_by

submissions = {name: pd.read_csv(config.PREDICTIONS_DIR / f"{name}.csv", parse_dates=["target_time_local"]) for name in MODELS}
test = holdout_frame(submissions)
test_names = ["persistence", "climatology", "raw_ecmwf", *MODELS]
test_mae = score_by(test, test_names)
display(test_mae.round(2))
pooled = score_by(test, test_names, overall="pooled").loc["all", test_names]
print("hours pooled:", pooled.round(2).to_dict())

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(test["target_time_local"], test["obs_temp_c"], color="black", lw=1.4, label="observed")
ax.plot(test["target_time_local"], test["raw_ecmwf"], color="#1baf7a", lw=1, label="raw ECMWF")
ax.plot(test["target_time_local"], test["lr_mos"], color="#4a3aa7", lw=1, label="lr_mos")
ax.plot(test["target_time_local"], test["linear_gam"], color="#008300", lw=1.6, label="linear_gam")
ax.set(title="Sep 17-30, 2026: forecasts vs observed (local time)", ylabel="°C")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "linear_gam_test_forecast_vs_observed.png", dpi=150)
plt.show()